In [ ]:
import pandas as pd 
alljoined = pd.read_parquet("../data/processed/parquet/alljoined_airlines.parquet")
alljoined.head()

In [ ]:
alljoined.shape

In [ ]:
alljoined.info()

In [ ]:
alljoined.columns

In [ ]:
alljoined.duplicated(
    subset=[
        'fl_date',
        'op_unique_carrier',
        'op_carrier_fl_num',
        'origin_airport_id',
        'dest_airport_id'
    ]
).sum()

In [ ]:
alljoined.groupby([
    'fl_date',
    'op_unique_carrier',
    'op_carrier_fl_num',
    'origin_airport_id',
    'dest_airport_id'
]).size().value_counts().head()

In [ ]:
alljoined = alljoined.drop_duplicates(
    subset=[
        'fl_date',
        'op_unique_carrier',
        'op_carrier_fl_num',
        'origin_airport_id',
        'dest_airport_id'
    ]
)

In [ ]:
alljoined.groupby([
    'fl_date',
    'op_unique_carrier',
    'op_carrier_fl_num',
    'origin_airport_id',
    'dest_airport_id'
]).size().value_counts().head()

In [ ]:
df_clean = alljoined.drop(columns=[
    'unnamed:_0',
    'year',
    'month',
    'dep_delay_new',
    'dep_del15',
    'dep_delay_group',
    'arr_delay_new',
    'arr_del15',
    'arr_delay_group'
])

In [ ]:
df_clean = df_clean.rename(columns={
    'fl_date': 'flight_date',
    'op_unique_carrier': 'airline_code',
    'op_carrier_fl_num': 'flight_number',
    'origin_airport_id': 'origin_airport_id',
    'dest_airport_id': 'destination_airport_id',
    'dep_delay': 'departure_delay',
    'arr_delay': 'arrival_delay',
    'cancelled': 'is_cancelled',
    'diverted': 'is_diverted'
})

In [ ]:
df_clean.columns

In [ ]:
#primary key test
df_clean.duplicated(subset=[
    'flight_date',
    'airline_code',
    'flight_number',
    'origin_airport_id',
    'destination_airport_id'
]).sum()

In [ ]:
df_clean[['flight_date','airline_code','flight_number',
          'origin_airport_id','destination_airport_id']].isnull().sum()

In [ ]:
df_clean[['departure_delay','arrival_delay']].isnull().sum()

In [ ]:
df_clean[(df_clean['is_cancelled'] == 0) & (df_clean['arrival_delay'].isnull())].shape

In [ ]:
df_clean = df_clean[
    ~(
        (df_clean['is_cancelled'] == 0) &
        (df_clean['arrival_delay'].isnull())
    )
]

In [ ]:
df_clean[
    (df_clean['is_cancelled'] == 0) &
    (df_clean['arrival_delay'].isnull())
].shape

In [ ]:
df_clean.dtypes

In [ ]:
df_clean['flight_date'] = pd.to_datetime(df_clean['flight_date'])

In [ ]:
df_clean['flight_date'].isnull().sum()

In [ ]:
df_clean['is_cancelled'] = df_clean['is_cancelled'].astype(bool)
df_clean['is_diverted'] = df_clean['is_diverted'].astype(bool)

In [ ]:
df_clean.dtypes

In [ ]:
df_clean[['departure_delay','arrival_delay']].describe()

In [ ]:
airline = pd.read_parquet("../data/processed/parquet/airline_key.parquet")
airline.head()

In [ ]:
airline = airline.drop(columns=['unnamed:_0'])
airline.head()

In [ ]:
airline = airline.rename(columns = {"op_unique_carrier":"airline_code"})

In [ ]:
airline.duplicated().sum()

In [ ]:
airline.duplicated(subset=['airline_code']).sum()

In [ ]:
airline.isnull().sum()

In [ ]:
airline.info()

In [ ]:
airline.to_parquet(
    "../data/processed/curated/airline_curated.parquet",
    index=False
)

In [ ]:
airport = pd.read_parquet("../data/processed/parquet/airport_info.parquet")
airport.head()

In [ ]:
airport = airport.rename(columns={
    'orgin_aiport_id': 'airport_id',
    'description': 'airport_name',
    'code.y': 'airport_code'
})

In [ ]:
airport.dtypes

In [ ]:
airport.head()

In [ ]:
airport.isnull().sum()


In [ ]:
airport = airport.dropna(subset=['airport_id'])

In [ ]:
airport.isnull().sum()

In [ ]:
airport['airport_id'] = airport['airport_id'].astype(int)

In [ ]:
airport.duplicated(subset=['airport_id']).sum()

In [ ]:
airport[airport.duplicated(subset=['airport_id'], keep=False)]\
    .sort_values('airport_id')

In [ ]:
airport = airport.drop_duplicates(subset = ['airport_id'], keep = 'first')

In [ ]:
airport.duplicated(subset=['airport_id']).sum()

In [ ]:
airport.to_parquet(
    "../data/processed/curated/airport_curated.parquet",
    index=False
)

In [ ]:
import pandas as pd 
wp = pd.read_parquet("../data/processed/parquet/weather.parquet")
wp.head()

In [ ]:
wp = wp.drop(columns=['time'])

In [ ]:
wp.shape

In [ ]:
wp.info()

In [ ]:
wp['date'].isnull().sum()

In [ ]:
wp['date'].duplicated().sum()

In [ ]:
wp['date'].min() , wp['date'].max()

In [ ]:
flight_delays = pd.read_parquet("../data/processed/curated/flight_delays_curated.parquet")
flight_delays.head()

In [ ]:
wp.head()

In [ ]:
df_final = flight_delays.merge(
wp,
left_on= 'flight_date',
right_on='date',
how='left')


In [ ]:
df_final.head()

In [ ]:
df_final.columns

In [ ]:
df_final.isnull().sum()

In [ ]:
flight_delays.isnull().sum()

In [ ]:
df_final = df_final.drop(columns=['date'])

In [ ]:
df_final.duplicated(subset=['flight_date','flight_number','origin_airport_id','destination_airport_id','airline_code']).sum()

In [ ]:
df_final = df_final.rename(columns={
    "temperature_2m_max": "max_temperature_c",
    "temperature_2m_min": "min_temperature_c",
    "precipitation_sum": "total_precipitation_mm",
    "snowfall_sum": "total_snowfall_mm",
    "rain_sum": "total_rain_mm",
})

In [ ]:
cols = [
    "flight_date",
    "airline_code",
    "flight_number",
    "origin_airport_id",
    "destination_airport_id",
    
    "departure_delay",
    "arrival_delay",
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay",
    
    "max_temperature_c",
    "min_temperature_c",
    "total_precipitation_mm",
    "total_snowfall_mm",
    "total_rain_mm"
]

df_final = df_final[cols]

In [ ]:
df_final.to_parquet("../data/processed/intermediate/flight_weather.parquet", index=False)

In [ ]:
holidays = pd.read_csv("../data/raw/us_holidays.csv")

print(holidays.head())

In [ ]:
holidays.shape

In [ ]:
holidays.isnull().sum()

In [ ]:
holidays.duplicated().sum()

In [ ]:
holidays.to_parquet("../data/processed/curated/holidays.parquet",index=False)

In [ ]:
df = pd.read_parquet("../data/processed/intermediate/flight_weather.parquet")
df.head()